# State and gradient across time

Prerequisite: the Python primer and Chapters 11–12. Run all cells from a fresh kernel. The standard-library cells need no model download, GPU, or API. We inspect supplied weights; we do not fit a translator or reproduce a paper benchmark. Code: Apache-2.0. Explanatory prose: CC BY-SA 4.0.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'code/part-iii').is_dir():
    root = next(p for p in root.parents if (p / 'code/part-iii').is_dir())
sys.path.insert(0, str(root / 'code/part-iii'))
from core import fixture, results
import json
import math
from fractions import Fraction
data = fixture()
observed = results(data)
print('Fixture:', data['version'], '| CPU | deterministic supplied values')

Fixture: part-iii-sequence-v1 | CPU | deterministic supplied values


## Three state updates

Predict the final state before running: a clue of one is halved twice, giving one quarter. The printed old state must equal the preceding new state. Parameters remain fixed.

In [2]:
for row in observed['short_rnn'][0]:
    print(json.dumps(row))
assert [r['state'][0] for r in observed['short_rnn'][0]] == [1, .5, .25]

{"step": 1, "token_id": 0, "input": [1, 0], "previous": [0, 0], "preactivation": [1.0, 0.0], "state": [1.0, 0.0], "logits": [1.125, 0.0], "probabilities": [0.7549149868676283, 0.24508501313237172], "choice": 0}
{"step": 2, "token_id": 3, "input": [0, 0], "previous": [1.0, 0.0], "preactivation": [0.5, 0.0], "state": [0.5, 0.0], "logits": [0.625, 0.0], "probabilities": [0.6513548646660542, 0.34864513533394575], "choice": 0}
{"step": 3, "token_id": 4, "input": [0, 0], "previous": [0.5, 0.0], "preactivation": [0.25, 0.0], "state": [0.25, 0.0], "logits": [0.375, 0.0], "probabilities": [0.5926665999540697, 0.4073334000459302], "choice": 0}


## Same long task, different failure mechanisms

The source fixture is the unchanged Chapter 10 collision set. Find the positive surviving blue coordinate, then compare it with the red output bias. Distinct RNN probabilities and identical window probabilities are different observations.

In [3]:
for ids, rows, window in zip(data['long_task']['prefix_ids'], observed['long_rnn'], observed['window']):
    print('input_ids:', ids, 'last_state:', rows[-1]['state'], 'rnn_choice:', rows[-1]['choice'], 'window:', window)
assert observed['long_rnn'][1][-1]['state'][1] == float(Fraction(1, 256))
assert observed['long_rnn'][1][-1]['choice'] == 0
assert observed['window'][0] == observed['window'][1]

input_ids: [0, 2, 2, 2, 2, 2, 2, 3, 4] last_state: [0.00390625, 0.0] rnn_choice: 0 window: {'visible_ids': [3, 4], 'state': [0, 0], 'logits': [0.125, 0.0], 'probabilities': [0.5312093733737563, 0.46879062662624377], 'choice': 0}
input_ids: [1, 2, 2, 2, 2, 2, 2, 3, 4] last_state: [0.0, 0.00390625] rnn_choice: 0 window: {'visible_ids': [3, 4], 'state': [0, 0], 'logits': [0.125, 0.0], 'probabilities': [0.5312093733737563, 0.46879062662624377], 'choice': 0}


## Backward rates and one shared parameter

The new diagnostic loss is half the squared distance of the final scalar state from one. It is not the output cross-entropy. The independent closed form is L(w) = (w² − 1)² / 2. Its derivative at one half is −3/4. Compare that with the sum of the three local-use contributions.

In [4]:
print(json.dumps(observed['bptt'], indent=2))
w = .5
closed_form_gradient = 2 * w * (w * w - 1)
assert closed_form_gradient == -3/4 == observed['bptt']['weight_gradient']
print('new_weight:', observed['updated_weight'], 'new_loss:', observed['after_update']['loss'])

{
  "states": [
    1.0,
    0.5,
    0.25
  ],
  "loss": 0.28125,
  "state_gradients": [
    -0.1875,
    -0.375,
    -0.75
  ],
  "weight_contributions": [
    -0.0,
    -0.375,
    -0.375
  ],
  "weight_gradient": -0.75
}
new_weight: 0.575 new_loss: 0.22403144531250005


## Distance table and gates

These powers are constructed sensitivity multipliers, not measured curves. Gates are supplied activations; their resulting cell and hidden values are computed. A forget factor near one still decays across sufficiently many steps.

In [5]:
for row in observed['distance_products']:
    print(row)
print(json.dumps(observed['gates'], indent=2))
assert observed['gates']['retention_099_hundred_edges'] < .37

{'edges': 0, 'contracting': 1.0, 'expanding': 1.0}
{'edges': 1, 'contracting': 0.5, 'expanding': 1.5}
{'edges': 2, 'contracting': 0.25, 'expanding': 2.25}
{'edges': 4, 'contracting': 0.0625, 'expanding': 5.0625}
{'edges': 8, 'contracting': 0.00390625, 'expanding': 25.62890625}
{'edges': 16, 'contracting': 1.52587890625e-05, 'expanding': 656.8408355712891}
{
  "lstm_cell": 0.8200000000000001,
  "lstm_hidden": 0.5063024061289559,
  "gru_candidate": 0.197375320224904,
  "gru_hidden": 0.7397375320224905,
  "retention_09_eight_edges": 0.4304672100000001,
  "retention_099_hundred_edges": 0.3660323412732292
}


## Transfer check with visible answer

Insert one filler into the short blue sequence. The final blue coordinate is 1/8, exactly equal to the red bias. The first-candidate tie rule therefore selects red. This is a decision-margin failure even in exact arithmetic; no floating-point underflow is needed.

In [6]:
from core import rnn_log
tied = rnn_log([1, 2, 3, 4], data['rnn'])[-1]
print(tied)
assert tied['logits'] == [.125, .125] and tied['choice'] == 0

{'step': 4, 'token_id': 4, 'input': [0, 0], 'previous': [0.0, 0.25], 'preactivation': [0.0, 0.125], 'state': [0.0, 0.125], 'logits': [0.125, 0.125], 'probabilities': [0.5, 0.5], 'choice': 0}
